In [15]:
from typing import TypedDict

from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_core.runnables import RunnableConfig

from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.runtime import Runtime
from langgraph.checkpoint.memory import InMemorySaver # In memory checkpointing for graph -> Short term memory
from langgraph.store.memory import InMemoryStore # Long term memory store for graph -> Long term memory

In [16]:
class AgentState(MessagesState):
  intent: str
  order_id: str
  preferred_language: str

In [17]:
class Context(TypedDict):
  user_id: str
  plan: str

In [18]:
def parse_request(state: AgentState, config: RunnableConfig) -> dict:
  user_text= str(state['messages'][-1].content).strip()
  thread_id= config['configurable']['thread_id']

  print(f"[Node1] thread_id: {thread_id}")
  print(f"[Node1] message_count: {len(state['messages'])}")
  print(f"[Node1] existing_order: {state.get('order_id')}")

  if user_text.startswith("My order_id is"):
    order_id= user_text.removeprefix("My order_id is ").strip()

    return {
      "order_id": order_id,
      "intent": "save_order"
    }
  
  if user_text == "Where is my order":
    return {
      "intent": "track_order"
    }
  
  if user_text.startswith("Remember my language "):
    return { "intent": "save_language" }
  
  if user_text == "Which language do I prefer?":
    return {"intent": "ask_language"}
  
  return {"intent": "unknown"}
  

In [19]:
def load_user_memory(
    state: AgentState,
    runtime: Runtime[Context]
) -> dict:
  user_id = runtime.context['user_id']
  plan = runtime.context['plan']

  # Load user memory from the long-term store
  store = runtime.store

  namespace = ("users", user_id)

  if state['intent'] == "save_language":
    message = str(state['messages'][-1].content).strip()
    language = message.removeprefix("Remember my language ").strip()

    store.put(
      namespace=namespace,
      key="preferred_language",
      value={"value": language}
    )

  memory = store.get(namespace, "preferred_language")
  language = (
    memory.value['value']
    if memory is not None else "not set"
  )

  return {
    "preferred_language": language
  }


In [20]:
def respond(state: AgentState, config: RunnableConfig, runtime: Runtime[Context]) -> dict:
  thread_id= config['configurable']['thread_id']
  user_id = runtime.context['user_id']
  intent = state['intent']

  if intent == "save_order":
    answer = f"Order ID {state['order_id']} saved for user {user_id} in thread {thread_id}."

  elif intent == "track_order":
    if state.get("order_id"):
      answer = f"Tracking order {state['order_id']} for user {user_id} in thread {thread_id}."
    else:
      answer = f"No order ID found for user {user_id}. Please provide your order ID."

  elif intent == "save_language":
    answer = f"Preferred language '{state['preferred_language']}' saved for user {user_id}."

  elif intent == "ask_language":
    answer = f"User {user_id}'s preferred language is '{state.get('preferred_language', 'not set')}'."

  else:
    answer = f"Sorry, I didn't understand that request, user {user_id}."

  return {
    "messages": [AIMessage(content=answer)]
  }


In [21]:
builder = StateGraph(AgentState , context_schema=Context)

builder.add_node("parse_request", parse_request)
builder.add_node("load_user_memory", load_user_memory)
builder.add_node("respond", respond)

builder.add_edge(START, "parse_request")
builder.add_edge("parse_request", "load_user_memory")
builder.add_edge("load_user_memory", "respond")
builder.add_edge("respond", END)

checkpointer = InMemorySaver() # In memory checkpointing for graph -> Short term memory
store = InMemoryStore()  # Long term memory store for graph -> Long term memory

graph = builder.compile(
  checkpointer=checkpointer,
  store=store
)

In [22]:
def chat(user_id, thread_id, message, plan="free"):

  print(f"\nUser {user_id} | Thread {thread_id}")
  print(f"Input: {message}")

  result = graph.invoke({
    "messages": [HumanMessage(content=message)]
  },
    config={'configurable': {"thread_id": thread_id}},
    context={"user_id": user_id, "plan": plan}
  )

# graph has been invoked, now we can get the response from the result
  response = result['messages'][-1].content
  print(f"Output: {response}")
  print(f"total messages in thread: {len(result['messages'])}")

  return result

In [23]:
chat("user123", "thread1", "My order_id is 12345" , plan="premium")


User user123 | Thread thread1
Input: My order_id is 12345
[Node1] thread_id: thread1
[Node1] message_count: 1
[Node1] existing_order: None
Output: Order ID 12345 saved for user user123 in thread thread1.
total messages in thread: 2


{'messages': [HumanMessage(content='My order_id is 12345', additional_kwargs={}, response_metadata={}, id='2aab66a2-cadc-4e83-870a-7bbe12ba4066'),
  AIMessage(content='Order ID 12345 saved for user user123 in thread thread1.', additional_kwargs={}, response_metadata={}, id='9e324d68-20c5-444d-a692-9e80a0903f1b', tool_calls=[], invalid_tool_calls=[])],
 'intent': 'save_order',
 'order_id': '12345',
 'preferred_language': 'not set'}